[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/37_uncertainty_and_calibration/04_conformal_prediction.ipynb)

# Bài 04 — Conformal prediction

**🎯 Mục tiêu**
- Biến **bất kỳ** mô hình nào thành khoảng/tập dự đoán có **đảm bảo độ phủ** $\ge 1 - \alpha$, không cần giả định phân phối — chỉ cần dữ liệu **hoán đổi được** (exchangeable).
- **Split conformal** cho hồi quy from scratch; kiểm chứng đảm bảo bằng mô phỏng.
- Độ phủ **biên** vs **có điều kiện**: CQR (conformalized quantile regression) và conformal **theo nhóm** (Mondrian).
- Conformal cho **phân loại**: tập dự đoán, kích thước tập thích ứng với độ khó.
- Khi giả định bị vi phạm: **trôi dữ liệu** làm mất đảm bảo.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/37_uncertainty_and_calibration

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression

from ml_course.datasets import make_house_prices
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore")

houses = make_house_prices(6000, seed=0)
X = pd.get_dummies(houses.drop(columns="price_billion_vnd"), columns=["district"]).astype(float)
y = houses.price_billion_vnd.values
ALPHA = 0.1                                                            # mục tiêu: phủ 90%
print(f"{len(X):,} căn nhà | giá: trung vị {np.median(y):.1f}, 90% trong [{np.quantile(y, 0.05):.1f}, {np.quantile(y, 0.95):.1f}] tỷ")

## 1. Split conformal: ý tưởng

1. Chia dữ liệu: **train** (fit mô hình bất kỳ $\hat f$) và **calibration** ($n$ điểm).
2. Tính **điểm không phù hợp** (nonconformity score) trên calibration: $s_i = |y_i - \hat f(x_i)|$.
3. Lấy $\hat q$ = điểm số nhỏ thứ $\lceil (n + 1)(1 - \alpha)\rceil$ trong các $s_i$ (tức phân vị mức $\lceil (n + 1)(1 - \alpha)\rceil / n$).
4. Khoảng dự đoán cho điểm mới: $[\hat f(x) - \hat q,\; \hat f(x) + \hat q]$.

**Đảm bảo** (Vovk et al., 2005): nếu calibration và điểm mới **hoán đổi được** (vd cùng phân phối, độc lập), thì $\mathbb P(y_{\text{mới}} \in \text{khoảng}) \ge 1 - \alpha$ — với **mọi** mô hình $\hat f$, mọi phân phối, mọi $n$. Lý do: điểm số của điểm mới có hạng "ngẫu nhiên đều" trong $n + 1$ điểm số. Hiệu chỉnh $(n + 1)$ là cái giá cho mẫu hữu hạn.

In [ ]:
def conformal_quantile(scores, alpha):
    """Điểm số nhỏ thứ ⌈(n + 1)(1 − α)⌉ trong n điểm số calibration (vô cùng nếu vượt n)."""
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    return np.inf if k > n else np.sort(scores)[k - 1]


def split(seed, n_train=3000, n_cal=1500):
    idx = np.random.default_rng(seed).permutation(len(X))
    return idx[:n_train], idx[n_train:n_train + n_cal], idx[n_train + n_cal:]


tr, ca, te = split(0)
model = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[tr], y[tr])
q = conformal_quantile(np.abs(y[ca] - model.predict(X.iloc[ca])), ALPHA)
pred = model.predict(X.iloc[te])
covered = (y[te] >= pred - q) & (y[te] <= pred + q)
print(f"q̂ = {q:.3f} tỷ | khoảng = dự đoán ± {q:.2f} tỷ | độ phủ trên {len(te):,} nhà test = {covered.mean():.1%}")

### Kiểm chứng đảm bảo bằng mô phỏng

Lặp 100 lần với các cách chia ngẫu nhiên khác nhau. Đảm bảo nói về **xác suất trên cả quá trình** (chọn tập calibration + điểm mới), nên độ phủ thực tế của từng lần dao động quanh 90%:

In [ ]:
coverages, widths = [], []
for seed in range(100):
    tr_s, ca_s, te_s = split(seed)
    m = HistGradientBoostingRegressor(max_iter=100, random_state=0).fit(X.iloc[tr_s], y[tr_s])
    q_s = conformal_quantile(np.abs(y[ca_s] - m.predict(X.iloc[ca_s])), ALPHA)
    p_s = m.predict(X.iloc[te_s])
    coverages.append(np.mean(np.abs(y[te_s] - p_s) <= q_s))
    widths.append(2 * q_s)
print(f"độ phủ qua 100 lần: trung bình {np.mean(coverages):.3f} | 5%–95%: [{np.quantile(coverages, 0.05):.3f}, {np.quantile(coverages, 0.95):.3f}] | "
      f"độ rộng TB {np.mean(widths):.2f} tỷ")
plt.figure(figsize=(6, 3.2))
plt.hist(coverages, bins=20)
plt.axvline(1 - ALPHA, color="k", ls=":", label="1 − α")
plt.xlabel("độ phủ trên tập test")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Phủ biên ≠ phủ có điều kiện

Đảm bảo là **trung bình trên mọi căn nhà**. Giá nhà có nhiễu **nhân** (căn đắt dao động nhiều tiền hơn), nhưng khoảng split conformal có **cùng độ rộng** cho mọi căn:

In [ ]:
expensive = y[te] > np.quantile(y, 0.8)
print(f"độ phủ — nhà đắt (top 20% giá): {covered[expensive].mean():.1%} | nhà còn lại: {covered[~expensive].mean():.1%} | chung: {covered.mean():.1%}")

Phủ 90% "trung bình" nhưng nhà đắt chỉ được phủ khoảng 2/3, nhà rẻ bị phủ thừa với khoảng quá rộng. Hai cách khắc phục:

### 2.1. CQR — Conformalized Quantile Regression

Romano, Patterson & Candès (2019): train hai mô hình **phân vị** $\hat q_{\alpha/2}(x)$, $\hat q_{1 - \alpha/2}(x)$ (loss pinball — khoảng tự **co giãn** theo $x$), rồi conformal hoá để có đảm bảo: điểm số $s_i = \max(\hat q_{\alpha/2}(x_i) - y_i,\; y_i - \hat q_{1-\alpha/2}(x_i))$, khoảng $[\hat q_{\alpha/2}(x) - \hat q,\; \hat q_{1-\alpha/2}(x) + \hat q]$.

In [ ]:
lower = HistGradientBoostingRegressor(loss="quantile", quantile=ALPHA / 2, random_state=0).fit(X.iloc[tr], y[tr])
upper = HistGradientBoostingRegressor(loss="quantile", quantile=1 - ALPHA / 2, random_state=0).fit(X.iloc[tr], y[tr])
cqr_scores = np.maximum(lower.predict(X.iloc[ca]) - y[ca], y[ca] - upper.predict(X.iloc[ca]))
q_cqr = conformal_quantile(cqr_scores, ALPHA)
lo_cqr, hi_cqr = lower.predict(X.iloc[te]) - q_cqr, upper.predict(X.iloc[te]) + q_cqr
covered_cqr = (y[te] >= lo_cqr) & (y[te] <= hi_cqr)
rows = []
for name, cov, width in [("split conformal", covered, np.full(len(te), 2 * q)), ("CQR", covered_cqr, hi_cqr - lo_cqr)]:
    rows.append({"phương pháp": name, "phủ chung": cov.mean(), "phủ nhà đắt": cov[expensive].mean(), "phủ nhà khác": cov[~expensive].mean(),
                 "rộng TB": width.mean(), "rộng TB nhà đắt": width[expensive].mean(), "rộng TB nhà khác": width[~expensive].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
order = np.argsort(pred)[::15]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (name, lo, hi) in zip(axes, [("split conformal", pred - q, pred + q), ("CQR", lo_cqr, hi_cqr)]):
    ax.vlines(pred[order], lo[order], hi[order], alpha=0.4)
    ax.scatter(pred[order], y[te][order], s=6, color="C3", label="giá thật")
    ax.plot([0, pred.max()], [0, pred.max()], "k:", lw=0.8)
    ax.set_xlabel("giá dự đoán (tỷ)")
    ax.set_title(name)
axes[0].set_ylabel("khoảng 90%")
axes[0].legend()
plt.tight_layout()
plt.show()

CQR giữ độ phủ chung 90% nhưng phân bổ **thông minh** hơn: rộng ở nhà đắt (nhiều bất định), hẹp ở nhà rẻ — độ phủ hai nhóm cân bằng hơn hẳn, trong khi độ rộng trung bình còn nhỏ hơn. Đảm bảo **có điều kiện** chính xác cho mọi $x$ là không thể nói chung (Vovk, 2012; Barber et al., 2021); CQR chỉ cải thiện xấp xỉ.

### 2.2. Conformal theo nhóm (Mondrian)

Nếu cần đảm bảo cho **từng nhóm định trước** (khu vực, nhóm khách hàng — gắn với công bằng, chương 36), calibrate **riêng** cho từng nhóm: mỗi nhóm có $\hat q_g$ riêng → đảm bảo $\ge 1 - \alpha$ trong **từng** nhóm (miễn mỗi nhóm đủ mẫu calibration).

In [ ]:
district_te, district_ca = houses.district.values[te], houses.district.values[ca]
res_ca = np.abs(y[ca] - model.predict(X.iloc[ca]))
rows = []
for d in sorted(houses.district.unique()):
    q_g = conformal_quantile(res_ca[district_ca == d], ALPHA)
    m_te = district_te == d
    rows.append({"khu vực": d, "n calibration": (district_ca == d).sum(), "phủ — chung q̂": covered[m_te].mean(),
                 "phủ — q̂ theo nhóm": np.mean(np.abs(y[te][m_te] - pred[m_te]) <= q_g), "q̂ nhóm": q_g})
print(f"q̂ chung = {q:.3f}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Với $\hat q$ chung, khu vực đắt (Quận 1, Quận 3) bị phủ thiếu, khu vực rẻ bị phủ thừa; calibrate theo nhóm đưa mọi khu vực về quanh 90% — khoảng rộng hơn ở nơi cần.

## 3. Conformal cho phân loại: tập dự đoán

Thay vì một nhãn, trả về **tập** nhãn đủ để chứa nhãn đúng với xác suất $\ge 1 - \alpha$. Điểm số đơn giản (LAC): $s_i = 1 - \hat p(y_i\mid x_i)$; tập dự đoán $C(x) = \{k: 1 - \hat p(k\mid x) \le \hat q\}$. Dữ liệu chữ số với 10% nhãn nhiễu (một số ảnh vốn mơ hồ).

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16
r = np.random.default_rng(0)
yd = yd.copy()
flip = r.random(len(yd)) < 0.1
yd[flip] = r.integers(0, 10, flip.sum())

cov_runs, size_runs = [], []
for seed in range(30):
    idx = np.random.default_rng(seed).permutation(len(Xd))
    d_tr, d_ca, d_te = idx[:900], idx[900:1350], idx[1350:]
    clf = LogisticRegression(max_iter=3000).fit(Xd[d_tr], yd[d_tr])
    q_cls = conformal_quantile(1 - clf.predict_proba(Xd[d_ca])[np.arange(len(d_ca)), yd[d_ca]], ALPHA)
    sets = (1 - clf.predict_proba(Xd[d_te])) <= q_cls
    cov_runs.append(sets[np.arange(len(d_te)), yd[d_te]].mean())
    size_runs.append(sets.sum(1))
    if seed == 0:
        top1 = (clf.predict(Xd[d_te]) == yd[d_te]).mean()
        sets0, labels0 = sets, yd[d_te]
print(f"accuracy top-1 = {top1:.3f} | độ phủ tập dự đoán qua 30 lần: TB {np.mean(cov_runs):.3f}, nhỏ nhất {np.min(cov_runs):.3f}")
sizes0 = sets0.sum(1)
print("phân bố kích thước tập (lần 0):", {int(k): int(v) for k, v in zip(*np.unique(sizes0, return_counts=True))})
print(f"độ phủ theo kích thước tập: tập 1 nhãn = {sets0[sizes0 == 1][np.arange((sizes0 == 1).sum()), labels0[sizes0 == 1]].mean():.1%} | "
      f"tập ≥ 2 nhãn = {sets0[sizes0 >= 2][np.arange((sizes0 >= 2).sum()), labels0[sizes0 >= 2]].mean():.1%}")

Phần lớn ảnh nhận tập **1 nhãn** (mô hình chắc chắn); ảnh mơ hồ nhận tập **nhiều nhãn** — kích thước tập là thước đo bất định dễ hiểu ("mô hình phân vân giữa 3 và 8"). Tập rỗng có thể xuất hiện khi không lớp nào đủ khả năng. Điểm số APS/RAPS (Romano et al., 2020; Angelopoulos et al., 2021) cho tập thích ứng tốt hơn LAC.

## 4. Khi giả định hoán đổi bị vi phạm

Đảm bảo chỉ đúng khi dữ liệu mới **giống** dữ liệu calibration. Giả sử thị trường tăng giá 15% sau khi triển khai (mô hình và $\hat q$ vẫn từ dữ liệu cũ):

In [ ]:
for inflation in [0.0, 0.05, 0.15]:
    y_shift = y[te] * (1 + inflation)
    print(f"giá tăng {inflation:4.0%}: độ phủ split conformal = {np.mean(np.abs(y_shift - pred) <= q):.1%} | CQR = {np.mean((y_shift >= lo_cqr) & (y_shift <= hi_cqr)):.1%}")

Đảm bảo sụp đổ khi phân phối trôi. Cách xử lý: **calibrate lại** trên dữ liệu gần đây nhất (chỉ cần ít dữ liệu có nhãn — rẻ hơn train lại mô hình), conformal **có trọng số** cho covariate shift (Tibshirani et al., 2019), conformal **thích ứng** cho chuỗi thời gian (Gibbs & Candès, 2021), và **giám sát** độ phủ thực tế sau triển khai (chương 40).

## 🧠 Kiểm tra nhanh

1. Viết các bước của split conformal. Vì sao cần hiệu chỉnh $(n + 1)$?
2. Đảm bảo của conformal cần giả định gì? Không cần giả định gì?
3. Độ phủ biên khác độ phủ có điều kiện thế nào? Ví dụ trong bài?
4. CQR và Mondrian conformal cải thiện độ phủ theo nhóm bằng cách nào?
5. Tập dự đoán phân loại được tạo ra sao? Kích thước tập nói lên điều gì?
6. Điều gì xảy ra với đảm bảo khi dữ liệu trôi? Khắc phục?

## 📌 Tóm tắt

- Conformal: điểm số không phù hợp trên tập calibration → phân vị hiệu chỉnh → khoảng/tập có độ phủ $\ge 1 - \alpha$ cho **mọi** mô hình, chỉ cần hoán đổi được.
- Đảm bảo là **biên**; CQR thích ứng độ rộng theo $x$, Mondrian đảm bảo theo nhóm.
- Phân loại: tập dự đoán, kích thước tập = độ bất định dễ hiểu.
- Trôi dữ liệu phá đảm bảo → calibrate lại, conformal có trọng số/thích ứng, giám sát độ phủ.

## 📚 Tài liệu tham khảo

- Angelopoulos & Bates (2023) — *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*
- Vovk, Gammerman & Shafer (2005) — *Algorithmic Learning in a Random World*
- Romano, Patterson & Candès (2019) — *Conformalized Quantile Regression*
- Tibshirani et al. (2019) — *Conformal Prediction Under Covariate Shift*
- Barber et al. (2021) — *The limits of distribution-free conditional predictive inference*
- [MAPIE](https://mapie.readthedocs.io/) — thư viện conformal cho scikit-learn